# Phase 6: Performance Evaluation

This notebook evaluates the 10 trained models using the three publication metrics: Jaccard Accuracy, Hamming Loss, and Penalty Matrix Score. It creates the comparative results table (replicating Table 6), plots a 12x12 confusion matrix heatmap for the best model (Random Forest), and prints a detailed per-class classification report (replicating Table 7).

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

sys.path.append(os.path.abspath('../'))
from src.features import scale_features
from src.models import load_models
from src.metrics import get_classification_metrics, load_penalty_matrix

# 1. Load datasets and perform identical split
df = pd.read_parquet('../data/interim/processed_features.parquet')
wells = df['WELL_ID'].unique()
df_train = df[df['WELL_ID'].isin(wells[:8])]
df_test = df[df['WELL_ID'].isin(wells[8:])]

original_cols = ['DEPTH_MD', 'CALI', 'RSHA', 'RMED', 'RDEP', 'RHOB', 'GR', 'NPHI', 'PEF', 'DTC', 'SP', 'BS']
wavelet_cols = original_cols + [f'{col}_CWT' for col in ['GR', 'NPHI', 'SP', 'RDEP', 'RHOB', 'DTC', 'PEF']]

X_train_orig, X_test_orig, _ = scale_features(df_train, df_test, original_cols)
X_train_wav, X_test_wav, _ = scale_features(df_train, df_test, wavelet_cols)

y_test = df_test['LITHOLOGY'].values
print(f'Test size: {len(y_test)}')

## 1. Replicating the Comparative Results Table

Loads both original and wavelet models, runs predictions on the test set, and builds the 5x3x2 table (replicating Table 6 from the paper).

In [ ]:
models_orig = load_models('original')
models_wav = load_models('wavelet')
penalty_mat = load_penalty_matrix()

results = []

for model_name in ['KNN', 'Decision Tree', 'Random Forest', 'XGBoost', 'LightGBM']:
    # Eval 12 feature model
    m_orig = models_orig[model_name]
    X_test_feat = X_test_orig if model_name == 'KNN' else df_test[original_cols]
    y_pred_orig = m_orig.predict(X_test_feat)
    metrics_orig = get_classification_metrics(y_test, y_pred_orig, penalty_mat)
    
    # Eval 19 feature model
    m_wav = models_wav[model_name]
    X_test_feat_wav = X_test_wav if model_name == 'KNN' else df_test[wavelet_cols]
    y_pred_wav = m_wav.predict(X_test_feat_wav)
    metrics_wav = get_classification_metrics(y_test, y_pred_wav, penalty_mat)
    
    results.append({
        'Model': model_name,
        'Accuracy (12)': metrics_orig['Accuracy'],
        'Penalty (12)': metrics_orig['PenaltyScore'],
        'Hamming (12)': metrics_orig['HammingLoss'],
        'Accuracy (19)': metrics_wav['Accuracy'],
        'Penalty (19)': metrics_wav['PenaltyScore'],
        'Hamming (19)': metrics_wav['HammingLoss']
    })

df_res = pd.DataFrame(results)
print('Models Comparison Table (Norway Replicated Table 6):')
df_res

## 2. 12x12 Confusion Matrix Heatmap

Visualizes class predictions vs ground truth for the best performing model (Random Forest).

In [ ]:
lithology_labels = [
    'Sandstone', 'Sandstone/Shale', 'Shale', 'Marl', 'Dolomite', 'Limestone',
    'Chalk', 'Halite', 'Anhydrite', 'Tuff', 'Coal', 'Basement'
]

# RF on 19 features prediction
rf_model = models_wav['Random Forest']
y_pred_rf = rf_model.predict(df_test[wavelet_cols])

cm = confusion_matrix(y_test, y_pred_rf, labels=range(12))

plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=lithology_labels, yticklabels=lithology_labels)
plt.xlabel('Predicted Lithology')
plt.ylabel('True Lithology')
plt.title('Confusion Matrix Heatmap — Best Model (Random Forest)', fontsize=14, pad=15)
plt.tight_layout()
plt.savefig('../plots/best_model_confusion_matrix.png', dpi=150)
plt.show()

## 3. Classification Report

Generates precision, recall, F1, and sample support for each of the 12 lithologies (replicating Table 7 from the paper).

In [ ]:
print('Per-Class Classification Report for Best Model (Random Forest 19 features):')
print(classification_report(y_test, y_pred_rf, labels=range(12), target_names=lithology_labels, zero_division=0))